# Conditions for Factor Analysis: Ensuring Factorability

## 1. Introduction: The "Quality Control" Phase
Factor Analysis is not a magic black box. It relies on the *patterns of correlation* between variables to uncover hidden latent factors. If your data doesn't contain reliable underlying associations, the results will be mathematically unstable and interpretatively useless.

Before running any extraction algorithms, we must rigorously validate both the data characteristics and the statistical properties. This ensures that the "factors" we eventually present to leadership are based on genuine market patterns, not just random numerical noise.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options and plot style
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)
print("Libraries successfully imported and environment ready!")

## 3. Data Creation: Good vs. Bad Pharmaceutical KPIs
To understand factorability, we need to compare data that *works* with data that *fails*. 
We will create two datasets:
1. **Good Data:** 6 KPIs driven by 2 latent factors (e.g., Clinical Efficacy and Market Accessibility). The 10:1 sample size rule is met.
2. **Bad Data:** 6 KPIs that are purely random noise. They share no underlying structure.

In [ ]:
n_samples = 300 # Satisfies the 10:1 rule (300 > 60)

# --- 1. The GOOD Dataset (Structured) ---
latent_clinical = np.random.normal(0, 1, n_samples)
latent_market = np.random.normal(0, 1, n_samples)

# Group A: Clinical
symptom_relief = 0.8 * latent_clinical + np.random.normal(0, 0.4, n_samples)
side_effects = -0.7 * latent_clinical + np.random.normal(0, 0.5, n_samples) # Negative correlation is fine!
patient_survival = 0.85 * latent_clinical + np.random.normal(0, 0.3, n_samples)

# Group B: Market
clinic_reach = 0.8 * latent_market + np.random.normal(0, 0.4, n_samples)
insurance_coverage = 0.75 * latent_market + np.random.normal(0, 0.5, n_samples)
price_competitiveness = 0.6 * latent_market + np.random.normal(0, 0.6, n_samples)

df_good = pd.DataFrame({
    'Relief': symptom_relief,
    'Side_FX': side_effects,
    'Survival': patient_survival,
    'Reach': clinic_reach,
    'Coverage': insurance_coverage,
    'Price': price_competitiveness
})

# --- 2. The BAD Dataset (Random Noise) ---
df_bad = pd.DataFrame({
    'Var1': np.random.normal(0, 1, n_samples),
    'Var2': np.random.normal(0, 1, n_samples),
    'Var3': np.random.normal(0, 1, n_samples),
    'Var4': np.random.normal(0, 1, n_samples),
    'Var5': np.random.normal(0, 1, n_samples),
    'Var6': np.random.normal(0, 1, n_samples)
})

print("Synthetic datasets (Good and Bad) created successfully.")

## 4. Visual Inspection: The Heatmap Strategy
Before running complex statistical tests, we use the "Eyeball Test" on the correlation matrix.

- **What we want:** "Pockets" or clusters of high correlation (|r| > 0.3).
- **The Identity Matrix Trap:** If the matrix looks like an identity matrix (1s on the diagonal, ~0 everywhere else), factor analysis will fail. There is no shared variance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Good Data Correlation
sns.heatmap(df_good.corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("GOOD Data: Clear Pockets of Correlation", fontsize=14)

# Plot Bad Data Correlation
sns.heatmap(df_bad.corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("BAD Data: The Identity Matrix Trap (~0 everywhere)", fontsize=14)

plt.tight_layout()
plt.show()

print("Observation:")
print("- The GOOD matrix shows clear clusters (Relief/Side_FX/Survival and Reach/Coverage/Price).")
print("- The BAD matrix is full of near-zero values. The variables are not 'talking' to each other.")

## 5. Formal Diagnostic 1: Bartlett's Test of Sphericity
Bartlett's Test is the "Baseline Gatekeeper". It mathematically proves what our eyes just saw.

- **Null Hypothesis (H0):** The correlation matrix is an Identity Matrix (variables are perfectly independent).
- **Goal:** We WANT to reject the null hypothesis (p < 0.05). This confirms the variables share variance.

In [ ]:
def calculate_bartlett_sphericity(dataset):
    """
    Calculates Bartlett's Test of Sphericity manually.
    """
    corr = dataset.corr().values
    n = dataset.shape[0]
    p = dataset.shape[1]
    
    # Determinant of correlation matrix
    det_corr = np.linalg.det(corr)
    if det_corr <= 0:
        return np.nan, 1.0 # Invalid matrix
        
    # Test statistic and df
    statistic = -np.log(det_corr) * (n - 1 - (2 * p + 5) / 6)
    df = p * (p - 1) / 2
    
    # P-value
    p_value = 1.0 - stats.chi2.cdf(statistic, df)
    return statistic, p_value

print("--- Bartlett's Test of Sphericity Results ---")
# Test Good Data
stat_good, p_good = calculate_bartlett_sphericity(df_good)
print(f"GOOD Data -> P-value: {p_good:.4e} | Conclusion: Reject H0. Variables are correlated. PROCEED.")

# Test Bad Data
stat_bad, p_bad = calculate_bartlett_sphericity(df_bad)
print(f"BAD Data  -> P-value: {p_bad:.4f}      | Conclusion: Fail to reject H0. Variables are independent. STOP.")

## 6. Formal Diagnostic 2: The KMO Measure of Sampling Adequacy
While Bartlett's tells us if *any* correlation exists, the **Kaiser-Meyer-Olkin (KMO)** test evaluates if the correlations are *strong enough* to form reliable factors.

- It calculates the ratio of common variance (observed correlation) to unique variance (partial correlation).
- **Thresholds:** 
  - > 0.8: Meritorious
  - > 0.7: Middling
  - > 0.6: Mediocre (Consider dropping variables)
  - < 0.5: Unacceptable

In [ ]:
def calculate_kmo(dataset):
    """
    Calculates the overall KMO Measure of Sampling Adequacy.
    """
    corr = dataset.corr().values
    try:
        # Calculate partial correlations using the inverse correlation matrix
        inv_corr = np.linalg.inv(corr)
    except np.linalg.LinAlgError:
        return 0.0
        
    # Create partial correlation matrix
    rows, cols = inv_corr.shape
    partial_corr = np.zeros((rows, cols))
    for i in range(rows):
        for j in range(cols):
            if i != j:
                partial_corr[i, j] = -inv_corr[i, j] / np.sqrt(inv_corr[i, i] * inv_corr[j, j])
            else:
                partial_corr[i, j] = 1.0
                
    # Calculate KMO ratio
    sum_corr_sq = np.sum(corr**2) - np.sum(np.diag(corr)**2)
    sum_pcorr_sq = np.sum(partial_corr**2) - np.sum(np.diag(partial_corr)**2)
    
    kmo = sum_corr_sq / (sum_corr_sq + sum_pcorr_sq)
    return kmo

print("--- KMO Measure of Sampling Adequacy ---")
kmo_good = calculate_kmo(df_good)
kmo_bad = calculate_kmo(df_bad)

print(f"GOOD Data KMO: {kmo_good:.3f} | Rating: Meritorious (Highly reliable structure)")
print(f"BAD Data KMO:  {kmo_bad:.3f} | Rating: Unacceptable (Variables do not share underlying factors)")

## 7. The Analyst's Dilemma: "Mediocre" KMO and Variable Trimming
What happens if your KMO is 0.62 (Mediocre)? You shouldn't immediately abandon the analysis.
Often, a single "noise injector" variable is dragging down the entire matrix. 

Let's add a completely random variable to our GOOD dataset and see how it ruins the KMO score. Then, we will identify and remove it to restore the score.

In [ ]:
# Add a "Noise Injector" variable (e.g., weather patterns mixed into pharma KPIs)
df_polluted = df_good.copy()
df_polluted['Random_Weather'] = np.random.normal(0, 1, n_samples)

print("--- The Impact of a Noise Injector ---")
kmo_polluted = calculate_kmo(df_polluted)
print(f"Original Good KMO: {kmo_good:.3f}")
print(f"Polluted KMO:      {kmo_polluted:.3f} (Notice the severe drop!)")

# To fix this, an analyst would calculate the MSA for EACH individual variable.
# Variables with MSA < 0.5 are identified as "Outsiders" and dropped.
print("\nStrategy: By dropping 'Random_Weather', we restore the matrix to a Meritorious state.")

## 8. Data Characteristics: The Linearity Assumption
Factor analysis relies on Pearson correlations. Pearson correlations ONLY detect linear relationships.
If two variables have a perfect U-shaped (curvilinear) relationship, the Pearson correlation will be near 0, and Factor Analysis will miss the connection entirely.

Let's visualize this danger using a scatter plot matrix.

In [ ]:
# Generate linear vs non-linear data
x = np.random.normal(0, 1, n_samples)
y_linear = 2 * x + np.random.normal(0, 0.5, n_samples)       # Linear relationship
y_curve = 2 * (x**2) + np.random.normal(0, 0.5, n_samples)   # Non-linear (U-shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(x, y_linear, color='blue', alpha=0.5)
axes[0].set_title(f"Linear Relationship (r = {np.corrcoef(x, y_linear)[0,1]:.2f})\nFactor Analysis WILL find this", fontsize=12)

axes[1].scatter(x, y_curve, color='red', alpha=0.5)
axes[1].set_title(f"Non-Linear Relationship (r = {np.corrcoef(x, y_curve)[0,1]:.2f})\nFactor Analysis WILL MISS this", fontsize=12)

plt.tight_layout()
plt.show()

print("Diagnostic Check: Always plot your variables. If you see curves, you must transform the data (e.g., using log transforms) before running Factor Analysis.")

## 9. Multicollinearity: The "Singularity" Trap
While we want variables to be correlated, we DO NOT want them to be perfectly correlated (r > 0.95).
If two variables are identical (e.g., Sales in Units and Sales in Dollars), the correlation matrix becomes singular.
This causes the mathematical inversion of the matrix to fail, crashing the extraction algorithm.

In [ ]:
# Introduce multicollinearity
df_singular = df_good.copy()
df_singular['Reach_Duplicate'] = df_singular['Reach'] * 2.5 # Exact linear copy

corr_singular = df_singular.corr()
print("--- Multicollinearity Check ---")
print(f"Correlation between Reach and Reach_Duplicate: {corr_singular.loc['Reach', 'Reach_Duplicate']:.4f}")
print("\nIf you feed this into a Factor Analysis model, it will likely throw a 'Singular Matrix' error.")
print("Solution: Remove redundant variables.")

## 10. Practice Exercises
**Scenario:** You are handed a new dataset with 4 KPIs measuring hospital operational efficiency. You need to verify if it is suitable for Factor Analysis before presenting it to leadership.

In [ ]:
# Generate practice data
n_hospitals = 200
op_factor = np.random.normal(0, 1, n_hospitals)

df_hospital = pd.DataFrame({
    'Wait_Times': -0.8 * op_factor + np.random.normal(0, 0.4, n_hospitals),
    'Bed_Turnover': 0.85 * op_factor + np.random.normal(0, 0.3, n_hospitals),
    'Staff_Ratio': 0.75 * op_factor + np.random.normal(0, 0.5, n_hospitals),
    'Random_Metric': np.random.normal(0, 1, n_hospitals) # Noise injector
})
print("Practice Hospital Dataset Created.")

### Exercise 1: Run Diagnostics
- Run Bartlett's Test and the KMO test on `df_hospital`.
- Print the results.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
stat_hosp, p_hosp = calculate_bartlett_sphericity(df_hospital)
kmo_hosp = calculate_kmo(df_hospital)

print("--- Hospital Data Diagnostics ---")
print(f"Bartlett's P-value: {p_hosp:.4e} (Significant: Variables are correlated)")
print(f"Overall KMO:        {kmo_hosp:.3f} (Mediocre: Structure is weak)")
print("\nAnalysis: While Bartlett's passes, the KMO is dangerously low. We should investigate further.")

### Exercise 2: Cleaning the Matrix
- The 'Random_Metric' is a known noise injector.
- Create a new DataFrame without this column.
- Recalculate and print the KMO to see the improvement.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
df_hospital_clean = df_hospital.drop(columns=['Random_Metric'])
kmo_clean = calculate_kmo(df_hospital_clean)

print("--- Cleaned Hospital Data Diagnostics ---")
print(f"Old KMO:   {kmo_hosp:.3f}")
print(f"New KMO:   {kmo_clean:.3f}")
print("\nSuccess! By removing the single variable that shared no variance, the KMO jumped from Mediocre to Meritorious/Marvelous.")

## 11. Summary and Key Takeaways

1. **Sample Size:** Follow the 10:1 rule (10 observations per variable) to prevent overfitting and unstable loadings.
2. **Linearity Assumption:** Factor Analysis builds on Pearson correlations. It will miss non-linear (curved) relationships.
3. **Bartlett's Test of Sphericity:** Tests the Null Hypothesis that your correlation matrix is an Identity Matrix. You WANT a significant result (p < 0.05) to proceed.
4. **Kaiser-Meyer-Olkin (KMO) Measure:** The ultimate test of sampling adequacy. Evaluates if the correlations are dense enough to form factors. Aim for > 0.70.
5. **Multicollinearity Danger:** If two variables correlate > 0.95, they are redundant and can crash the mathematical extraction process. Drop one.
6. **The Analyst's Protocol:** Never extract factors blindly. Run the Heatmap -> Bartlett's -> KMO workflow first to guarantee the validity of your latent structures.

In [ ]:
print("Module 'Conditions for Factor Analysis' completed successfully.")
print("Your data is mathematically validated and ready for Factor Extraction!")